# Swiss Law RAG: Retrieval Evaluation

This notebook measures whether a multilingual E5 retriever finds the expected Fedlex passage for 40 hand-written questions in German, English and French.

**Resources:** [Embedding model](https://huggingface.co/intfloat/multilingual-e5-small) · [Answer model](https://huggingface.co/Qwen/Qwen2.5-7B-Instruct) · [Source register](https://github.com/ChadSaglam/swiss-law-rag/blob/main/sources.json) · [Project code](https://github.com/ChadSaglam/swiss-law-rag)

The corpus contains six official provisions from the Swiss Code of Obligations and Value Added Tax Act. This is a retrieval test, not an evaluation of generated legal answers. The bundled texts are a dated snapshot and not legal or tax advice.

## 1. Setup

Internet access is required to install the embedding library, download the model and clone the project. A GPU is optional.

In [ ]:
!pip install -q sentence-transformers
!git clone -q https://github.com/ChadSaglam/swiss-law-rag.git
%cd swiss-law-rag

## 2. Run the fixed evaluation set

The script uses the same `Retriever` class as the Gradio app. It indexes 900-character chunks with a 150-character overlap and writes the complete result to `evaluation.json`.

In [ ]:
!python run_eval.py

## 3. Inspect the results

Hit@1 requires the expected source to be the first result. Hit@4 accepts it anywhere in the four passages sent to the answer model.

In [ ]:
import json
import pandas as pd

with open("evaluation.json", encoding="utf-8") as file:
    report = json.load(file)

summary = pd.DataFrame([
    {"language": "all", "questions": report["questions"], "hit_at_4": report["hit_at_4"]},
    *[
        {"language": language, "questions": values["questions"], "hit_at_4": values["hit_at_4"]}
        for language, values in report["by_language"].items()
    ],
])
summary["hit_at_4"] = summary["hit_at_4"].map("{:.1%}".format)
print(f"Hit@1: {report['hit_at_1']:.1%}")
display(summary)
print(f"Failed queries: {len(report['failures'])}")

## 4. Inspect one retrieval

Change the question to see which passages would be sent to the LLM. The source file and character offsets are not shown here; the app resolves the filenames to canonical Fedlex links through `sources.json`.

In [ ]:
from rag import Retriever

retriever = Retriever()
question = "How long must accounting records be retained?"

for index, chunk in enumerate(retriever.search(question, k=4), start=1):
    print(f"[{index}] {chunk.source}")
    print(chunk.text[:350], "...\n")

## Scope and limitations

The 40 questions were written around the six bundled provisions, so a high hit rate should not be read as general legal retrieval performance. The evaluation does not score answer correctness, citation faithfulness or inference-provider availability. Laws can change; use the canonical Fedlex links to check the current consolidated text.